# 💬 Messages: The Unit of Context

## Learning Objectives
In this notebook, you will learn:
1. **Text vs. message prompts** - when a bare string is enough and when you need message objects
2. **The four message types** - `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage`
3. **Message metadata** - `name`, `id`, and what they are actually for
4. **`usage_metadata`** - reading token counts off a response to track cost
5. **Hand-building history** - inserting AI and tool messages as if the model had produced them

## Prerequisites
- Completed `1-langchainintro.ipynb` through `3-tools.ipynb`
- `pip install langchain langchain-openai python-dotenv`
- A `.env` file with `OPENAI_API_KEY`

---
## 💡 Part 1: What Is a Message?

Messages are the fundamental unit of context for models in LangChain. They represent both the
input and the output of a model, carrying the content *and* the metadata needed to represent
the state of a conversation.

Every message object contains:

- **Role** — identifies the message type (system, user, assistant, tool)
- **Content** — the actual payload: text, images, audio, documents
- **Metadata** — optional fields such as response info, message IDs, and token usage

### Key Insight:
LangChain provides one standard message type that works across **all** providers. OpenAI,
Anthropic, and Google each wire messages differently on the API; you write against the
LangChain types and the integration package translates. Swapping providers does not change
your message-handling code.

---
## 🔑 Part 2: Environment Setup

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load credentials and initialize the model
# ============================================================================
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "❌ Set OPENAI_API_KEY in your .env file"

model = init_chat_model("gpt-4.1")

print(f"✅ Environment loaded — using {type(model).__name__}")

---
## 📝 Part 3: Text Prompts

The simplest input is a plain string. LangChain wraps it into a single `HumanMessage` for you.

**Use text prompts when:**
- You have a single, standalone request
- You do not need conversation history
- You want minimal code complexity

In [ ]:
# ============================================================================
# TEXT PROMPT: A bare string is wrapped into a HumanMessage automatically
# ============================================================================
model.invoke("Please tell what is artificial intelligence")

In [ ]:
# ============================================================================
# TEXT PROMPT: Same shape, reading just the text back
# ============================================================================
response = model.invoke("what is langchain")
print("🤖", response.content[:400], "...")

---
## 🧱 Part 4: Message Prompts — the Four Types

For anything beyond a one-shot question you pass a **list of message objects**. There are four
types, and each has a distinct job:

| Type | Role | Purpose |
|------|------|---------|
| `SystemMessage` | system | Tells the model how to behave; sets tone, role, and guidelines |
| `HumanMessage` | user | User input — text, images, audio, files, any multimodal content |
| `AIMessage` | assistant | Model output — text content, tool calls, and provider metadata |
| `ToolMessage` | tool | The result of a single tool execution, fed back to the model |

> **Note**: the list *is* the conversation state. There is no hidden memory — whatever you pass
> is exactly what the model sees, which is why message construction is the core skill here.

### 4.1 ⚙️ System Message

A `SystemMessage` is an initial set of instructions that primes the model's behaviour. Use it
to set the tone, define the model's role, and establish guidelines for responses.

In [ ]:
# ============================================================================
# SYSTEM MESSAGE: Priming the model's role
# ============================================================================
from langchain.messages import AIMessage, HumanMessage, SystemMessage

messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem on artificial intelligence"),
]

response = model.invoke(messages)
print("🤖", response.content)

In [ ]:
# ============================================================================
# SYSTEM MESSAGE: Same question, a different role -> a different answer
# ============================================================================
system_msg = SystemMessage("You are a helpful coding assistant.")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?"),
]

response = model.invoke(messages)
print("🤖", response.content[:600], "...")

#### 🎨 Making the System Message Do More Work

The more specific the system message, the less steering each individual request needs. Compare
the output below with the terse version above — same user question, but the instructions now
pin down expertise level, format, and verbosity.

In [ ]:
# ============================================================================
# SYSTEM MESSAGE: Detailed instructions shape format, not just tone
# ============================================================================
system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?"),
]

response = model.invoke(messages)
print("🤖", response.content[:800], "...")

### 4.2 👤 Human Message and Metadata

A `HumanMessage` represents user input. Beyond `content`, two optional fields matter in real
applications:

- **`name`** — identifies *which* user spoke, in a multi-user or multi-persona conversation
- **`id`** — a stable identifier you choose, so a message can be traced through logs and
  observability tooling

> **Note**: these are metadata for *your* systems. Most providers ignore `id` entirely, and
> support for `name` varies — never rely on it to carry instructions to the model.

In [ ]:
# ============================================================================
# MESSAGE METADATA: Optional name and id fields
# ============================================================================
human_msg = HumanMessage(
    content="Hello!",
    name="alice",   # Optional: identify different users in one conversation
    id="msg_123",   # Optional: unique identifier for tracing
)

print(f"📋 content: {human_msg.content}")
print(f"📋 name:    {human_msg.name}")
print(f"📋 id:      {human_msg.id}")

In [ ]:
# ============================================================================
# INVOCATION: Metadata rides along; the model answers the content
# ============================================================================
response = model.invoke([human_msg])
response

### 4.3 🤖 AI Message

An `AIMessage` is the output of a model invocation. It can carry multimodal data, tool calls,
and provider-specific metadata.

You can also **construct one yourself** and insert it into the history — which is how you
replay a stored conversation, or seed the model with a response you want it to treat as its
own. The model cannot tell the difference.

In [ ]:
# ============================================================================
# AI MESSAGE: Hand-built and inserted into the conversation history
# ============================================================================
ai_msg = AIMessage("I'd be happy to help you with that question!")

messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,                              # inserted as if the model had said it
    HumanMessage("Great! What's 2+2?"),
]

response = model.invoke(messages)
print("🤖", response.content)

#### 📊 Token Usage

Every response carries `usage_metadata` with the token counts for that call. This is the
number to log if you care about cost — input and output tokens are billed at different rates,
so the split matters.

In [ ]:
# ============================================================================
# TOKEN USAGE: input / output / total for the call above
# ============================================================================
response.usage_metadata

### 4.4 🔧 Tool Message

For models that support tool calling, an `AIMessage` can contain **tool calls**. A
`ToolMessage` then passes the result of a single tool execution back to the model.

### Key Concepts:
- **`tool_call_id` must match**: the `ToolMessage`'s `tool_call_id` has to equal the `id` of
  the `AIMessage` tool call it answers. That pairing is how the model matches results to
  requests when several tools were called at once — mismatch it and the provider errors out.
- **`content=[]` on the AI message**: a pure tool-call turn has no text, which is exactly the
  "empty AI message" you saw in notebook 1.

Below we hand-build the whole exchange rather than running a real tool, so the structure is
visible in one cell.

In [ ]:
# ============================================================================
# TOOL MESSAGE: Hand-building an AIMessage -> ToolMessage exchange
# ============================================================================
from langchain.messages import ToolMessage

# The model's tool-call turn: no text content, only a request.
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123",
    }],
)

# What executing that tool produced.
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123",  # ⚠️ must match the call ID above
)

messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,     # the model's tool call
    tool_message,   # the tool's result
]

response = model.invoke(messages)  # the model now answers from the result
print("🤖", response.content)

In [ ]:
# ============================================================================
# INSPECT: The assembled conversation
# ============================================================================
messages

In [ ]:
# ============================================================================
# INSPECT: The ToolMessage on its own
# ============================================================================
tool_message

In [ ]:
# ============================================================================
# INSPECT: The final AIMessage produced from the tool result
# ============================================================================
response

---
## 📝 Summary

In this notebook, we learned:

### 1. Two Input Shapes
- **Text prompt**: a bare string, wrapped into a `HumanMessage` — fine for one-shot requests
- **Message list**: required as soon as you need a system prompt or conversation history

### 2. The Four Message Types
- **`SystemMessage`**: behaviour, role, and guidelines — the most leveraged message you write
- **`HumanMessage`**: user input, optionally tagged with `name` and `id`
- **`AIMessage`**: model output — text, tool calls, provider metadata
- **`ToolMessage`**: one tool's result, keyed back to its call

### 3. The List *Is* the State
- **No hidden memory**: the model sees exactly the list you pass, nothing more
- **You can forge history**: hand-built `AIMessage`s replay stored conversations seamlessly

### 4. Metadata Worth Reading
- **`usage_metadata`**: input/output/total tokens — log this to track cost
- **`tool_call_id`**: must match between the `AIMessage` tool call and its `ToolMessage`

### Next Steps
- **`5-structuredoutput.ipynb`** — force responses into a schema with Pydantic, `TypedDict`,
  and dataclasses via `with_structured_output` and `response_format`